# Delta Lake tables 
Use this notebook to explore Delta Lake functionality

In [1]:
from pyspark.sql.types import StructType, IntegerType, StringType, DoubleType

# define the schema
schema = StructType() \
    .add("ProductID", IntegerType(), True) \
    .add("ProductName", StringType(), True) \
    .add("Category", StringType(), True) \
    .add("ListPrice", DoubleType(), True)

df = spark.read.format("csv").option("header", "true").schema(schema).load("Files/products/products.csv")
# df now is a Spark DataFrame containing CSV data from "Files/products/products.csv"
display(df)

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 3, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 42456b05-1b33-4167-9688-002584502823)

In [2]:
df.write.format("delta").saveAsTable("managed_products")

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 4, Finished, Available, Finished)

In [3]:
df.write.format("delta").saveAsTable(
    "external_products", 
    path="abfss://90854aff-f850-4ffa-b6a9-fabadbf173dc@onelake.dfs.fabric.microsoft.com/3a8480a8-8055-4b87-a6d4-f399bcc7acd2/Files/external_products"
)

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 5, Finished, Available, Finished)

In [4]:
%%sql
DESCRIBE FORMATTED managed_products;

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 6, Finished, Available, Finished)

<Spark SQL result set with 12 rows and 3 fields>

In [5]:
%%sql
DESCRIBE FORMATTED external_products;

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 7, Finished, Available, Finished)

<Spark SQL result set with 12 rows and 3 fields>

In [6]:
%%sql
DROP TABLE managed_products;
DROP TABLE external_products;

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 9, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
%%sql
CREATE TABLE products
USING DELTA
LOCATION 'Files/external_products';

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 10, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [8]:
%%sql
SELECT * FROM products;

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 11, Finished, Available, Finished)

<Spark SQL result set with 295 rows and 4 fields>

In [9]:
%%sql
UPDATE products
SET ListPrice = ListPrice * 0.9
WHERE Category = 'Mountain Bikes';

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 12, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 1 fields>

In [10]:
%%sql
DESCRIBE HISTORY products;

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 13, Finished, Available, Finished)

<Spark SQL result set with 2 rows and 15 fields>

In [11]:
delta_table_path = 'Files/external_products'

current_data = spark.read.format("delta").load(delta_table_path)
display(current_data)

original_data = spark.read.format("delta").option("versionAsOf", 0).load(delta_table_path)
display(original_data)


StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 14, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 2d55c172-9f71-4aa7-8b88-d7c6eb259c2a)

SynapseWidget(Synapse.DataFrame, 2a3a0cbc-0869-4439-b17e-177f7acbbb85)

In [12]:
%%sql
-- Create a temporary view
CREATE OR REPLACE TEMPORARY VIEW products_view AS
    SELECT 
        Category, 
        COUNT(*) AS NumProducts, 
        MIN(ListPrice) AS MinPrice, 
        MAX(ListPrice) AS MaxPrice, 
        AVG(ListPrice) AS AvgPrice
    FROM products
    GROUP BY Category;

SELECT *
FROM products_view
ORDER BY Category;

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 16, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 37 rows and 5 fields>

In [13]:
%%sql
SELECT Category, NumProducts
FROM products_view
ORDER BY NumProducts DESC
LIMIT 10;

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 17, Finished, Available, Finished)

<Spark SQL result set with 10 rows and 2 fields>

In [14]:
from pyspark.sql.functions import col, desc

df_products = spark.sql("SELECT Category, MinPrice, MaxPrice, AvgPrice FROM products_view").orderBy(col("AvgPrice").desc())
display(df_products.limit(6))


StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 18, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 7f93b94f-d3f5-4211-8ea5-238199ed1396)

In [15]:
from notebookutils import mssparkutils
from pyspark.sql.types import *
from pyspark.sql.functions import *

inputPath = 'Files/data/'
mssparkutils.fs.mkdirs(inputPath)

jsonSchema = StructType([
    StructField("device", StringType(), False),
    StructField("status", StringType(), False)
])

iotstream = spark.readStream.schema(jsonSchema).option("maxFilesPerTrigger", 1).json(inputPath)

device_data = '''{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}'''

mssparkutils.fs.put(inputPath + "data.txt", device_data, True)

print("Source stream created...")

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 19, Finished, Available, Finished)

Source stream created...


In [16]:
delta_stream_table_path = 'Tables/iotdevicedata'
checkpointpath = 'Files/delta/checkpoint'

deltastream = iotstream.writeStream \
    .format("delta") \
    .option("checkpointLocation", checkpointpath) \
    .start(delta_stream_table_path)

print("Streaming to delta sink...")

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 20, Finished, Available, Finished)

Streaming to delta sink...


In [19]:
%%sql
SELECT * FROM IotDeviceData;

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 23, Finished, Available, Finished)

<Spark SQL result set with 16 rows and 2 fields>

In [18]:
more_data = '''{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}'''

mssparkutils.fs.put(inputPath + "more-data.txt", more_data, True)


StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 22, Finished, Available, Finished)

True

In [20]:
deltastream.stop()

StatementMeta(, 1a4bf5e1-08e2-4728-a27c-3bb29c61e4e7, 24, Finished, Available, Finished)